In [1]:
# ============================================================
# CELL 1: Mount Drive & Cài thư viện
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets transformers scikit-learn tqdm Pillow

Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2: Import & Config
# ============================================================
import os, json, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
from datetime import datetime

warnings.filterwarnings("ignore")

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR   = "/content/drive/MyDrive/BERT_IFND"
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5
MAX_LEN    = 256

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Device  : {DEVICE}")
print(f"Save dir: {SAVE_DIR}")

Device  : cuda
Save dir: /content/drive/MyDrive/BERT_IFND


In [3]:
# ============================================================
# CELL 3: Load dataset
# ============================================================
print("Đang tải dataset Nhat243/IFND-multimodal ...")
raw = load_dataset("Nhat243/IFND-multimodal")
print(raw)
print("\nFeatures:", raw["train"].features)
print("\nMẫu đầu:")
for k, v in raw["train"][0].items():
    print(f"  {k}: {str(v)[:100]}")

test_splits = [k for k in raw.keys() if k not in ("train", "validation")]
if not test_splits:
    test_splits = ["validation"]
print(f"\nTest splits: {test_splits}")

# Thống kê label
for split in ["train", "validation"]:
    labels  = raw[split]["label"]
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\n[{split}] Label distribution:")
    for u, c in zip(unique, counts):
        name = "Fake" if u == 0 else "Real"
        print(f"  Label {u} ({name}): {c:,} mẫu")

Đang tải dataset Nhat243/IFND-multimodal ...


README.md:   0%|          | 0.00/613 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/435M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/432M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/112M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1052 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1053 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 8416
    })
    validation: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1052
    })
    test: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1053
    })
})

Features: {'id': Value('int64'), 'text': Value('string'), 'image': Image(mode=None, decode=True), 'label': Value('int64')}

Mẫu đầu:
  id: 3851
  text: HP health minister urges people who have recovered from Covid to visit isolation wards & boost moral
  image: <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=450x250 at 0x787ACF501D90>
  label: 1

Test splits: ['test']

[train] Label distribution:
  Label 0 (Fake): 1,969 mẫu
  Label 1 (Real): 6,447 mẫu

[validation] Label distribution:
  Label 0 (Fake): 246 mẫu
  Label 1 (Real): 806 mẫu


In [8]:
# ============================================================
# CELL 4 (FIX): Dataset class — tắt decode ảnh hoàn toàn
# ============================================================
import datasets as ds

class IFNDBertDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len=MAX_LEN):
        self.tokenizer = tokenizer
        self.max_len   = max_len

        # Tắt decode ảnh vì BERT không dùng ảnh
        self.data = hf_dataset.cast_column(
            "image",
            ds.Image(decode=False)
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # TEXT
        text = (item.get("text")    or item.get("caption") or
                item.get("title")   or item.get("content") or "")
        enc  = self.tokenizer(
            str(text),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # LABEL
        label = int(item.get("label", item.get("labels", 0)))

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            enc["token_type_ids"].squeeze(0),
            label
        )

In [5]:
# ============================================================
# CELL 5: Model BERT + Classifier
# ============================================================
class BERTFakeNewsClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        hidden    = self.bert.config.hidden_size  # 768
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, token_type_ids):
        out     = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        cls_out = out.last_hidden_state[:, 0, :]  # [CLS] token
        return self.classifier(cls_out)

In [9]:
# ============================================================
# CELL 6: Khởi tạo tokenizer, dataloader
# ============================================================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_ds = IFNDBertDataset(raw["train"],      tokenizer)
val_ds   = IFNDBertDataset(raw["validation"], tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f"Train : {len(train_ds):,} mẫu | {len(train_loader):,} steps/epoch")
print(f"Val   : {len(val_ds):,} mẫu   | {len(val_loader):,} steps/epoch")
for s in test_splits:
    print(f"Test [{s}]: {len(raw[s]):,} mẫu")

Train : 8,416 mẫu | 526 steps/epoch
Val   : 1,052 mẫu   | 66 steps/epoch
Test [test]: 1,053 mẫu


In [10]:
# ============================================================
# CELL 7: Training + lưu latest checkpoint
# ============================================================
# Class weight xử lý mất cân bằng (Fake: 23%, Real: 77%)
num_fake = 1969
num_real = 6447
total    = num_fake + num_real
w_fake   = total / (2 * num_fake)
w_real   = total / (2 * num_real)
class_weights = torch.tensor([w_fake, w_real], dtype=torch.float).to(DEVICE)
print(f"Class weights → Fake: {w_fake:.4f} | Real: {w_real:.4f}")

model     = BERTFakeNewsClassifier().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

LATEST_CKPT = os.path.join(SAVE_DIR, "latest_epoch.pth")

# Resume nếu có checkpoint
start_epoch = 1
history     = []
if os.path.exists(LATEST_CKPT):
    print("Tìm thấy checkpoint, đang resume...")
    ckpt        = torch.load(LATEST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    history     = ckpt.get("history", [])
    print(f"Tiếp tục từ epoch {start_epoch} | "
          f"Val Acc: {ckpt['val_acc']:.4f} | Val F1: {ckpt['val_f1']:.4f}")

# Hàm evaluate
def evaluate(loader, split_name="Val"):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for ids, mask, token_type_ids, labels in tqdm(loader,
                                                      desc=f"  Eval [{split_name}]",
                                                      leave=False):
            logits     = model(ids.to(DEVICE),
                               mask.to(DEVICE),
                               token_type_ids.to(DEVICE))
            loss       = criterion(logits, labels.to(DEVICE))
            total_loss += loss.item()
            all_preds.extend(torch.argmax(logits, 1).cpu().tolist())
            all_labels.extend(labels.tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    print(f"  [{split_name}] Loss: {total_loss/len(loader):.4f} "
          f"| Acc: {acc:.4f} | F1: {f1:.4f}")
    return total_loss / len(loader), acc, f1, all_preds, all_labels

# Training loop
for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss              = 0
    train_preds, train_labels = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for ids, mask, token_type_ids, labels in loop:
        optimizer.zero_grad()
        logits = model(ids.to(DEVICE),
                       mask.to(DEVICE),
                       token_type_ids.to(DEVICE))
        loss   = criterion(logits, labels.to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item()
        train_preds.extend(torch.argmax(logits, 1).cpu().tolist())
        train_labels.extend(labels.tolist())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc  = accuracy_score(train_labels, train_preds)
    train_f1   = f1_score(train_labels, train_preds, average="macro")
    print(f"\nEpoch {epoch} Train → "
          f"Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

    val_loss, val_acc, val_f1, _, _ = evaluate(val_loader, "Val")
    scheduler.step()

    history.append({
        "epoch":      epoch,
        "train_loss": round(train_loss, 4),
        "train_acc":  round(train_acc,  4),
        "train_f1":   round(train_f1,   4),
        "val_loss":   round(val_loss,   4),
        "val_acc":    round(val_acc,    4),
        "val_f1":     round(val_f1,     4),
    })

    # Lưu latest checkpoint (ghi đè sau mỗi epoch)
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "val_acc":         val_acc,
        "val_f1":          val_f1,
        "history":         history,
    }, LATEST_CKPT)
    print(f"  ✅ latest_epoch.pth đã lưu (epoch {epoch})")

print("\n=== TRAINING HOÀN TẤT ===")

Class weights → Fake: 2.1371 | Real: 0.6527


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/5 [Train]: 100%|██████████| 526/526 [06:14<00:00,  1.41it/s, loss=0.0023]



Epoch 1 Train → Loss: 0.1169 | Acc: 0.9750 | F1: 0.9653


  [Val] Loss: 0.0573 | Acc: 0.9838 | F1: 0.9779
  ✅ latest_epoch.pth đã lưu (epoch 1)


...


Epoch 4 Train → Loss: 0.0090 | Acc: 0.9987 | F1: 0.9982


  [Val] Loss: 0.0529 | Acc: 0.9933 | F1: 0.9908
  ✅ latest_epoch.pth đã lưu (epoch 4)


Epoch 5/5 [Train]: 100%|██████████| 526/526 [06:21<00:00,  1.38it/s, loss=0.0002]



Epoch 5 Train → Loss: 0.0049 | Acc: 0.9994 | F1: 0.9992


  [Val] Loss: 0.0638 | Acc: 0.9924 | F1: 0.9894
  ✅ latest_epoch.pth đã lưu (epoch 5)

=== TRAINING HOÀN TẤT ===


In [11]:
# ============================================================
# CELL 8: Đánh giá test splits
# ============================================================
ckpt = torch.load(LATEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✅ Load epoch {ckpt['epoch']} | "
      f"Val Acc: {ckpt['val_acc']:.4f} | Val F1: {ckpt['val_f1']:.4f}\n")

timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
all_results = {}

for split in test_splits:
    print(f"{'='*55}")
    print(f"Split: {split}  ({len(raw[split]):,} mẫu)")
    print('='*55)

    dl = DataLoader(
        IFNDBertDataset(raw[split], tokenizer),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    preds_all, labels_all = [], []
    with torch.no_grad():
        for ids, mask, token_type_ids, labels in tqdm(dl, desc=split):
            logits = model(ids.to(DEVICE),
                           mask.to(DEVICE),
                           token_type_ids.to(DEVICE))
            preds_all.extend(torch.argmax(logits, 1).cpu().tolist())
            labels_all.extend(labels.tolist())

    acc    = accuracy_score(labels_all, preds_all)
    f1     = f1_score(labels_all, preds_all, average="macro")
    f1_w   = f1_score(labels_all, preds_all, average="weighted")
    report = classification_report(labels_all, preds_all,
                                   target_names=["Fake (0)", "Real (1)"])
    print(report)
    print(f"Accuracy: {acc:.4f} | F1 Macro: {f1:.4f} | F1 Weighted: {f1_w:.4f}\n")

    all_results[split] = {
        "accuracy":    float(acc),
        "f1_macro":    float(f1),
        "f1_weighted": float(f1_w),
        "report":      classification_report(
                           labels_all, preds_all,
                           target_names=["Fake (0)", "Real (1)"],
                           output_dict=True)
    }

# Bảng tóm tắt
print(f"\n{'='*67}")
print("TỔNG KẾT TEST SPLITS")
print(f"{'='*67}")
print(f"{'Split':<35} | {'Accuracy':>8} | {'F1 Macro':>8} | {'F1 Weighted':>11}")
print("-"*67)
for split, res in all_results.items():
    print(f"{split:<35} | {res['accuracy']:>8.4f} | "
          f"{res['f1_macro']:>8.4f} | {res['f1_weighted']:>11.4f}")

✅ Load epoch 5 | Val Acc: 0.9924 | Val F1: 0.9894

Split: test  (1,053 mẫu)


test: 100%|██████████| 66/66 [00:17<00:00,  3.87it/s]


              precision    recall  f1-score   support

    Fake (0)       1.00      0.98      0.99       246
    Real (1)       0.99      1.00      1.00       807

    accuracy                           1.00      1053
   macro avg       1.00      0.99      0.99      1053
weighted avg       1.00      1.00      1.00      1053

Accuracy: 0.9953 | F1 Macro: 0.9933 | F1 Weighted: 0.9952


TỔNG KẾT TEST SPLITS
Split                               | Accuracy | F1 Macro | F1 Weighted
-------------------------------------------------------------------
test                                |   0.9953 |   0.9933 |      0.9952


In [12]:
# ============================================================
# CELL 9: Lưu tất cả kết quả
# ============================================================
# 1. Weights model
weights_path = os.path.join(SAVE_DIR, "bert_weights_final.pth")
torch.save(model.state_dict(), weights_path)
print(f"✅ Weights         : {weights_path}")

# 2. Full checkpoint kèm timestamp
full_ckpt_path = os.path.join(SAVE_DIR, f"bert_full_{timestamp}.pth")
torch.save({
    "epoch":           ckpt["epoch"],
    "model_state":     model.state_dict(),
    "optimizer_state": ckpt["optimizer_state"],
    "scheduler_state": ckpt["scheduler_state"],
    "val_acc":         ckpt["val_acc"],
    "val_f1":          ckpt["val_f1"],
    "history":         ckpt["history"],
}, full_ckpt_path)
print(f"✅ Full checkpoint : {full_ckpt_path}")

# 3. JSON đầy đủ
json_data = {
    "timestamp":        timestamp,
    "checkpoint_epoch": int(ckpt["epoch"]),
    "val_acc":          float(ckpt["val_acc"]),
    "val_f1":           float(ckpt["val_f1"]),
    "training_history": ckpt["history"],
    "test_results":     all_results,
    "config": {
        "model":      "bert-base-uncased",
        "dataset":    "Nhat243/IFND-multimodal",
        "epochs":     EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr":         LR,
        "max_len":    MAX_LEN,
        "class_weights": {"Fake": round(w_fake, 4), "Real": round(w_real, 4)},
    }
}
json_path = os.path.join(SAVE_DIR, f"results_all_{timestamp}.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)
print(f"✅ JSON report     : {json_path}")

# 4. TXT dễ đọc
txt_path = os.path.join(SAVE_DIR, f"results_all_{timestamp}.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"BERT | IFND-multimodal | {timestamp}\n")
    f.write("=" * 67 + "\n")
    f.write(f"Model        : bert-base-uncased (text-only)\n")
    f.write(f"Dataset      : Nhat243/IFND-multimodal\n")
    f.write(f"Epochs       : {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}\n")
    f.write(f"Class weights: Fake={w_fake:.4f} | Real={w_real:.4f}\n")
    f.write(f"Checkpoint   : epoch {ckpt['epoch']}\n")
    f.write(f"Val Acc      : {ckpt['val_acc']:.4f}\n")
    f.write(f"Val F1 Macro : {ckpt['val_f1']:.4f}\n")
    f.write("\n--- Training History ---\n")
    f.write(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | "
            f"{'Train F1':>8} | {'Val Loss':>8} | {'Val Acc':>7} | {'Val F1':>7}\n")
    f.write("-" * 75 + "\n")
    for h in ckpt["history"]:
        f.write(f"{h['epoch']:>6} | {h['train_loss']:>10.4f} | {h['train_acc']:>9.4f} | "
                f"{h['train_f1']:>8.4f} | {h['val_loss']:>8.4f} | "
                f"{h['val_acc']:>7.4f} | {h['val_f1']:>7.4f}\n")
    f.write(f"\n--- Test Results ---\n")
    f.write(f"{'Split':<35} | {'Accuracy':>8} | {'F1 Macro':>8} | {'F1 Weighted':>11}\n")
    f.write("-" * 67 + "\n")
    for split, res in all_results.items():
        f.write(f"{split:<35} | {res['accuracy']:>8.4f} | "
                f"{res['f1_macro']:>8.4f} | {res['f1_weighted']:>11.4f}\n")
print(f"✅ TXT report      : {txt_path}")

# 5. Liệt kê tất cả file đã lưu
print(f"\n{'='*65}")
print(f"TẤT CẢ FILE ĐÃ LƯU tại: {SAVE_DIR}")
print(f"{'='*65}")
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1e6
    print(f"  {fname:<50} {size:>7.1f} MB")

✅ Weights         : /content/drive/MyDrive/BERT_IFND/bert_weights_final.pth
✅ Full checkpoint : /content/drive/MyDrive/BERT_IFND/bert_full_20260408_133618.pth
✅ JSON report     : /content/drive/MyDrive/BERT_IFND/results_all_20260408_133618.json
✅ TXT report      : /content/drive/MyDrive/BERT_IFND/results_all_20260408_133618.txt

TẤT CẢ FILE ĐÃ LƯU tại: /content/drive/MyDrive/BERT_IFND
  bert_full_20260408_133618.pth                       1311.7 MB
  bert_weights_final.pth                               438.8 MB
  latest_epoch.pth                                    1311.7 MB
  results_all_20260408_133618.json                       0.0 MB
  results_all_20260408_133618.txt                        0.0 MB
